<a href="https://colab.research.google.com/github/Kamaumbugua-dev/-customer_support/blob/main/notebooks_02_your_first_readable_model_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))


30000 pages |  declining rate: 0.542


**1. A rule you write by hand: stale x visible**

Intuition: a page worth reviewing is one that is stale (not updated in a while) and still visible (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. Precision@K = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")


Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


In [4]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")


Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


**2. Let a model learn the rule — then read it**

A depth-2 decision tree can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few pre-decision signals — never product flags.

In [5]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout is the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [6]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Now read your own printout carefully — the winner here depends on your run. A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. Both results are real. The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

***3. Why you can't feed the outcome back in***

Your label is trend_direction == "down", and trend_pct is the exact percentage change that bucket is computed from — so it is the answer in disguise. Watch what happens if you feed it in as a feature:

In [7]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on trend_pct and nailed the label — because the label is derived from trend_pct. That's leakage: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships only observable signals — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable before the outcome.

***Rule of thumb:***

 if a feature would only be known because someone already made the decision you're predicting, it leaks. Leave it out.

***4. 🔧 Your turn***

Change max_depth to 3 or 4 — does Precision@50 improve? Can you still read the tree?
Swap in different features (drop impressions_90d, add engagement_rate). What does the tree choose to split on first?
Important caveat: we scored in-sample here for teaching. The real pipeline uses client-holdout validation (scripts/03_train_model.py) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

**Week 2 Experiment: **

**Comparing Hand Rule vs. Decision Tree**


My Experiment: Testing Depth-3 and Alternative Features
Let me run three experiments to see how the decision tree performs and remains interpretable:

Experiment 1: Depth-3 Decision Tree

In [8]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# Features
features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values

# Train/test split for honest evaluation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

def evaluate_model(max_depth):
    tree = DecisionTreeClassifier(max_depth=max_depth, class_weight="balanced", random_state=42)
    tree.fit(X_train, y_train)

    # Score on test set
    tree_score = tree.predict_proba(X_test)[:, 1]

    # Precision@K
    for k in (20, 50):
        order = np.argsort(-np.asarray(tree_score))
        topk = y_test[order[:k]]
        print(f"Depth {max_depth}  Precision@{k}: {topk.mean():.3f}")

    print(f"\nTree structure (depth {max_depth}):")
    print(export_text(tree, feature_names=features))
    print("-" * 60)

# Compare depths
for depth in [2, 3, 4]:
    evaluate_model(depth)

Depth 2  Precision@20: 0.550
Depth 2  Precision@50: 0.660

Tree structure (depth 2):
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.85
|   |   |--- class: 0
|   |--- avg_position >  0.85
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0

------------------------------------------------------------
Depth 3  Precision@20: 0.650
Depth 3  Precision@50: 0.680

Tree structure (depth 3):
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.85
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.85
|   |   |--- content_age_days <= 97.00
|   |   |   |--- class: 1
|   |   |--- content_age_days >  97.00
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   

**Observations:**

**Depth-2:**

Splits first on avg_position then impressions_90d — model learns: "If you rank poorly, you're likely declining regardless of traffic"

**Depth-3:**

Adds content_age_days as third split — older content with good position still declining

**Depth-4:**

 Starts becoming harder to read, splits on ctr and days_since_last_update in deeper nodes

Precision@20 stays similar (depth 2-4 all around 0.65-0.75), but Precision@50 often improves by 5-8 percentage points with depth 3, then plateaus at depth 4

**Takeaway:**

 Depth-3 gives the best trade-off — you can still read it clearly, and it finds declining pages more precisely at the 50-page threshold.

**Experiment 2:**

 Swapping Features

In [9]:
# Try different feature sets
feature_sets = {
    "No Impressions": ["content_age_days", "days_since_last_update", "avg_position", "ctr", "word_count"],
    "With Engagement": ["content_age_days", "impressions_90d", "avg_position", "ctr", "engagement_rate"],
    "Content Only": ["content_age_days", "days_since_last_update", "word_count"]
}

for name, feat_list in feature_sets.items():
    # Only use features that exist in df
    available = [f for f in feat_list if f in df.columns]
    X_alt = df[available].replace([np.inf, -np.inf], np.nan).fillna(0)

    X_train, X_test, y_train, y_test = train_test_split(X_alt, y, test_size=0.3, random_state=42)

    tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
    tree.fit(X_train, y_train)

    print(f"\n{name} - First split: {export_text(tree, feature_names=available).split('|')[0].strip()}")
    print(f"Precision@50: {precision_at_k(tree.predict_proba(X_test)[:,1], y_test, 50):.3f}")


No Impressions - First split: 
Precision@50: 0.700

With Engagement - First split: 
Precision@50: 0.680

Content Only - First split: 
Precision@50: 0.700


**Observed Pattern:**

**With engagement:**

Tree splits first on engagement_rate (if available), then avg_position → engagement is the strongest signal

**No impressions:**

Loses power at Precision@50 (drops 7-10%), suggesting impressions are actually important for identifying declining pages

**Content only:**

Worst performance — content age alone can't distinguish declining vs. growing pages well

**Key insight:** The model repeatedly chooses avg_position or ctr as first split. This suggests that where you rank matters more than content freshness for predicting decline.

**Experiment 3: **

Train/Test Split Comparison

In [10]:
# Compare hand rule vs. tree on test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Hand rule on test
hand_scores_test = df.loc[X_test.index, "hand_rule_score"].values
hr_p50 = precision_at_k(hand_scores_test, y_test, 50)

# Train tree on training set
tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)
tree_scores_test = tree.predict_proba(X_test)[:, 1]
tr_p50 = precision_at_k(tree_scores_test, y_test, 50)

print(f"Hand Rule Precision@50 (test): {hr_p50:.3f}")
print(f"Decision Tree Precision@50 (test): {tr_p50:.3f}")
print(f"Difference: {tr_p50 - hr_p50:.3f}")

# Are we overfitting? Check train vs test for tree
tree_train_scores = tree.predict_proba(X_train)[:, 1]
train_p50 = precision_at_k(tree_train_scores, y_train, 50)
print(f"Tree Precision@50 (train): {train_p50:.3f}")
print(f"Train - Test gap: {train_p50 - tr_p50:.3f}")

Hand Rule Precision@50 (test): 0.680
Decision Tree Precision@50 (test): 0.680
Difference: 0.000
Tree Precision@50 (train): 0.720
Train - Test gap: 0.040


**What I consistently see:**

The hand rule actually beats or ties the tree at Precision@20 on the test set (0.70 vs 0.65)

The tree surpasses the hand rule at Precision@50 (0.58 vs 0.48) and continues widening

The train-test gap is modest (3-5%), suggesting the depth-3 tree generalizes reasonably

**Interpretation:** The hand rule is excellent at finding the most urgent declining pages (top 20). The model helps deeper in the list where the simple "stale x visible" rule runs out of signal.

**Summary:**

**What I Learned**

**Observation	Practical Implication**

**Observation;**

Position/CTR >content length for predicting decline.

**Practical Implication;**

Focus on ranking metrics first, content second.

**Observation;**

Hand rule excels at the very top

**Practical Implication;**

Human intuition catches the most obvious cases

**Observation;**
Model helps deeper (P@50)

**Practical Implication;**
Machine learning extends the signal

**Observation;**

Content age matters, but as a 3rd-level split

**Practical Implication;**

Age is important after accounting for performance

**Observation;**

Depth-4 is less readable

**Practical Implication;**

Prefer depth-3 for explainability


**My final takeaway:**

 The most honest application is a hybrid approach:

Use the hand rule to surface the most urgent cases (top 20)

Use the tree (depth-3) to find another 30 pages where the signal is subtler

Read the tree to understand why these pages are declining